In [4]:
!pip install git+https://github.com/FelixKrones/nnUNet.git

  Cloning https://github.com/FelixKrones/nnUNet.git to /tmp/pip-req-build-1i3g_w89
  Running command git clone --filter=blob:none --quiet https://github.com/FelixKrones/nnUNet.git /tmp/pip-req-build-1i3g_w89
  Resolved https://github.com/FelixKrones/nnUNet.git to commit e0e5193f7f07cdb6fa5c99a25281a6d547f49edc
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached acvl_utils-0.2.6.tar.gz (34 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of matplotlib to determine which version is compatible with other require

In [7]:
!sed -i 's/torch.load(filename_or_checkpoint, map_location=self.device)/torch.load(filename_or_checkpoint, map_location=self.device, weights_only=False)/g' /usr/local/lib/python3.11/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py

In [8]:
import nnunetv2
print("OK")

OK


In [ ]:
# the dataset used here was named nnunet-raw... replace it with what you are using
import os
os.environ["nnUNet_raw"] = "/kaggle/input/nnunet-raw/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"
print("done")


done


In [ ]:
import json
#replace with your path to dataset.json
with open("/kaggle/input/nnunet-raw/nnUNet_raw/Dataset001_ECG/dataset.json") as f:
    data = json.load(f)

data


# Training

In [ ]:
# Re-run preprocessing
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

In [ ]:
!nnUNetv2_train 1 2d 4 --c
# 4 is representative of the 4th fold... replace it with the current fold number you are training

# Ensembled Prediction

In [12]:
!nnUNetv2_predict \
  -i "/kaggle/input/datasets/fusychutney/name-grey" \
  -o "/kaggle/working/testing" \
  -d 1 \
  -c 2d \
  -f 0 1 2 3 4


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 1 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1 cases that I would like to predict

Predicting gray:
perform_everything_on_device: True
100%|█████████████████████████████████████████████| 4/4 [00:00<00:00, 35.65it/s]
sending off prediction to background worker for resampling and export
done with gray


# Manual Ensembling

In [ ]:
!nnUNetv2_ensemble \
    -i /kaggle/working/output-fold0 \
    /kaggle/working/output-fold1 \
    /kaggle/working/output-fold2 \
    /kaggle/working/output-fold3 \
    /kaggle/working/output-fold4 \
    -o /kaggle/working/output-ensemble-1

#combining already obtained predictions from individual models

# Evaluation

In [ ]:
!nnUNetv2_evaluate_folder \
  -djfile /kaggle/working/nnUNet_preprocessed/Dataset001_ECG/dataset.json \
  -pfile /kaggle/working/nnUNet_preprocessed/Dataset001_ECG/nnUNetPlans.json \
  /kaggle/input/nnunet-raw/nnUNet_raw/Dataset001_ECG/labelsTs \ 
  /kaggle/working/output-ensemble \
  -o /kaggle/working/eval_fold.json

#replace with your ground truth and prediction files
